# Name That LLM classifier comparison

Compare existing Mixed-training logistic-regression and random-forest classifiers on a limited MG-only Name That LLM cohort. The texts were scored with `gemma_3_270m_it`; this notebook evaluates 32, 64, and 128-token feature windows and does not run language-model inference or full-text evaluation.

Because the cohort contains MG texts only, results are MG scores, predictions, detection rates, and agreement diagnostics rather than balanced classification metrics.

## Dataset profile

Compare the complete raw `train.csv` cohort with all available processed Stage-1 texts. Word counts use whitespace splitting.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from notebooks.utils.report_diagrams_utils import save_png_figure

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
PLOT_OUTPUT_ROOT = DATA_ROOT / 'plots' / 'compare_name_that_llm_classifiers'
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT = DATA_ROOT / 'processed'
DATASET = 'name_that_llm_train'
SCORER_LLM = 'gemma_3_270m_it'
def with_word_count(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result['source_llm'] = result['source_llm'].astype(str).str.strip()
    result['word_count'] = result['text'].fillna('').astype(str).map(lambda text: len(text.split()))
    return result

all_texts = with_word_count(
    pd.read_csv(DATA_ROOT / 'raw' / 'name-that-llm' / 'train.csv', usecols=['text', 'model'])
    .rename(columns={'model': 'source_llm'})
)

source_metadata = pd.read_csv(
    DATA_ROOT / 'raw' / 'name-that-llm' / 'name_that_llm_train_MG.csv',
    usecols=['id', 'source_llm'], dtype={'id': str},
).assign(id=lambda frame: frame['id'].str.strip())

processed_paths = sorted((PROCESSED_ROOT / f'{DATASET}__{SCORER_LLM}' / 'MG' / 'data').glob('*.parquet'))
if not processed_paths:
    raise FileNotFoundError(f'no processed Stage-1 files found under {PROCESSED_ROOT}')
processed_texts = pd.concat(
    (pd.read_parquet(path, columns=['id', 'text', 'tokens']) for path in processed_paths),
    ignore_index=True,
)
processed_texts['id'] = processed_texts['id'].astype(str).str.strip()
processed_texts = processed_texts.drop_duplicates('id').merge(
    source_metadata, on='id', how='left', validate='one_to_one',
)
if processed_texts['source_llm'].isna().any():
    raise ValueError('processed texts are missing source LLM metadata')
processed_texts = with_word_count(processed_texts)
processed_texts['token_count'] = processed_texts['tokens'].map(len)

In [ ]:
def count_by_source(frame: pd.DataFrame, dataset_view: str) -> pd.DataFrame:
    counts = frame.groupby('source_llm', as_index=False).agg(text_count=('source_llm', 'size'))
    counts['dataset_view'] = dataset_view
    total = pd.DataFrame([{
        'source_llm': 'ALL', 'text_count': len(frame), 'dataset_view': dataset_view,
    }])
    return pd.concat([counts, total], ignore_index=True)[['dataset_view', 'source_llm', 'text_count']]

dataset_counts = pd.concat([
    count_by_source(all_texts, 'all'),
    count_by_source(processed_texts, 'processed'),
], ignore_index=True)
dataset_counts.sort_values(['dataset_view', 'source_llm'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_frames = [
    ('all', 'per source LLM', all_texts, 'source_llm'),
    ('all', 'aggregated', all_texts.assign(source_llm='ALL'), 'source_llm'),
    ('processed', 'per source LLM', processed_texts, 'source_llm'),
    ('processed', 'aggregated', processed_texts.assign(source_llm='ALL'), 'source_llm'),
]
max_word_count = max(frame['word_count'].max() for _, _, frame, _ in plot_frames)
bins = range(0, int(max_word_count) + 2)
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=False)
for axis, (dataset_view, grouping, frame, hue) in zip(axes.flat, plot_frames):
    sns.histplot(data=frame, x='word_count', hue=hue, bins=bins, element='step', stat='count', common_norm=False, ax=axis)
    axis.set_title(f'{dataset_view.title()} texts: {grouping}')
    axis.set_xlabel('Text length (words)')
    axis.set_ylabel('Text count')
    axis.set_xlim(0, max_word_count + 1)
plt.tight_layout()
save_png_figure(fig, PLOT_OUTPUT_ROOT / 'dataset_profile.png')
plt.show()

## Processed token-length profile

Token lengths come from the stored `tokens` produced by the scoring LLM tokenizer.

In [ ]:
token_plot_frames = [
    ('per source LLM', processed_texts, 'source_llm'),
    ('aggregated', processed_texts.assign(source_llm='ALL'), 'source_llm'),
]
max_token_count = max(frame['token_count'].max() for _, frame, _ in token_plot_frames)
token_bins = range(0, int(max_token_count) + 2)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=False)
for axis, (grouping, frame, hue) in zip(axes, token_plot_frames):
    sns.histplot(data=frame, x='token_count', hue=hue, bins=token_bins, element='step', stat='count', common_norm=False, ax=axis)
    axis.set_title(f'Processed texts: {grouping}')
    axis.set_xlabel('Text length (tokens)')
    axis.set_ylabel('Text count')
    axis.set_xlim(0, max_token_count + 1)
plt.tight_layout()
save_png_figure(fig, PLOT_OUTPUT_ROOT / 'token_profile.png')
plt.show()

In [ ]:
from __future__ import annotations

import itertools
import os
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

from notebooks.utils.classification_experiment_utils import discover_model_artifacts
from notebooks.utils.feature_aggregation_utils import FeatureSet, ensure_prepared_partition, load_prepared_partition, prepared_partition_path
from notebooks.utils.model_training_utils import load_model_artifact, load_model_metadata, score_feature_vector_model

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
PROCESSED_ROOT = DATA_ROOT / 'processed'
DATASET = 'name_that_llm_train'
SCORER_LLM = 'gemma_3_270m_it'
TRAINING_COMPOSITION = 'Kaggle+MAGE+Outfox'
RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
LIMIT = 1800
MIN_TOKEN_COUNT = 250
WINDOWS = (32, 64, 128)
WINDOWS = (32, 64)

MODEL_FAMILIES = ('logistic_regression', 'random_forest')
STRIDE = 16
FEATURE_SET = FeatureSet('simple_similarity', (
    'mean_candidate_similarity__mean', 'mean_candidate_similarity__std',
    'top20_coverage__mean', 'top20_coverage__std',
    'observed_logprob__mean', 'observed_logprob__std',
    'mean_candidate_logprob_delta__mean', 'mean_candidate_logprob_delta__std',
))
MODELS_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'models'
print({'data_root': DATA_ROOT, 'windows': WINDOWS, 'limit': LIMIT, 'models': MODEL_FAMILIES})

In [ ]:
def expected_prepared_metadata(window: int) -> dict[str, object]:
    return {
        'scorer_llm': SCORER_LLM, 'dataset': DATASET, 'split': 'MG',
        'feature_set': FEATURE_SET.name, 'feature_columns': list(FEATURE_SET.source_columns),
        'window_configuration': str(window), 'schema_version': 1, 'stride': STRIDE,
    }

def load_prepared_window(window: int, *, on_text_prepared: Callable[[], None] | None = None) -> pd.DataFrame:
    status = ensure_prepared_partition(
        DATA_ROOT, PROCESSED_ROOT, scorer_llm=SCORER_LLM, dataset=DATASET, split='MG',
        feature_set=FEATURE_SET, window_configuration=window, stride=STRIDE,
        on_text_prepared=on_text_prepared,
    )
    if status.get('status') != 'available':
        raise RuntimeError(f'Name That LLM preparation unavailable for {window}: {status.get("error", status)}')
    path = prepared_partition_path(DATA_ROOT, scorer_llm=SCORER_LLM, dataset=DATASET, split='MG', feature_set=FEATURE_SET, window_configuration=window)
    return load_prepared_partition(path, expected=expected_prepared_metadata(window))

source_csv = DATA_ROOT / 'raw' / 'name-that-llm' / 'name_that_llm_train_MG.csv'
source_metadata = pd.read_csv(source_csv, dtype={'id': str, 'source_id': str})
required_metadata = {'id', 'source_id', 'prompt_id', 'source_llm', 'origin', 'source_dataset'}
if not required_metadata.issubset(source_metadata.columns):
    raise ValueError(f'missing source metadata columns: {sorted(required_metadata - set(source_metadata.columns))}')
source_metadata['text_id'] = source_metadata['id'].astype(str).str.strip()
if source_metadata['text_id'].duplicated().any():
    raise ValueError('source CSV contains duplicate IDs')

prepared = {}
for window in tqdm(WINDOWS, desc='Preparing feature windows', unit='window'):
    with tqdm(desc=f'{window} texts', unit='text', leave=False) as progress:
        prepared[window] = load_prepared_window(window, on_text_prepared=progress.update)
available_ids_by_window = {
    window: set(prepared[window]['text_id'].astype(str).unique()) for window in WINDOWS
}
common_ids = set.intersection(*available_ids_by_window.values())
available_ids = sorted(common_ids, key=int)
if len(available_ids) < LIMIT:
    counts = {window: len(ids) for window, ids in available_ids_by_window.items()}
    raise ValueError(f'expected at least {LIMIT} texts common to all windows, found {len(available_ids)}; per-window counts: {counts}')
selected_text_ids = available_ids[-LIMIT:]

def selected_window(window: int) -> pd.DataFrame:
    frame = prepared[window].copy()
    frame['text_id'] = frame['text_id'].astype(str)
    frame = frame.loc[frame['text_id'].isin(selected_text_ids)].copy()
    if set(frame['text_id']) != set(selected_text_ids):
        raise ValueError(f'window {window} does not contain the complete selected cohort')
    if not frame['split'].eq('MG').all() or not frame['label'].eq(1).all():
        raise ValueError(f'window {window} contains non-MG rows')
    return frame.merge(source_metadata, on='text_id', how='left', validate='many_to_one')

window_frames = {window: selected_window(window) for window in WINDOWS}
pd.DataFrame({window: {'feature_rows': len(frame), 'text_count': frame['text_id'].nunique()} for window, frame in window_frames.items()}).T

In [ ]:
def find_model_artifact(model_family: str, window: int) -> Path:
    matches = discover_model_artifacts(MODELS_ROOT, filters={
        'run_id': RUN_ID, 'model_family': model_family, 'scorer_llm': SCORER_LLM,
        'feature_set': FEATURE_SET.name, 'training_regime': TRAINING_COMPOSITION,
        'training_window': str(window),
    })
    if len(matches) != 1:
        raise ValueError(f'expected one artifact for {model_family}/{window}, found: {matches}')
    return matches[0]

def score_configuration(window: int, model_family: str) -> pd.DataFrame:
    artifact_path = find_model_artifact(model_family, window)
    metadata = load_model_metadata(artifact_path, expected={
        'run_id': RUN_ID, 'model_family': model_family, 'scorer_llm': SCORER_LLM,
        'feature_set': FEATURE_SET.name, 'training_regime': TRAINING_COMPOSITION,
        'training_window': str(window),
    })
    if metadata['feature_columns'] != list(FEATURE_SET.source_columns):
        raise ValueError(f'artifact feature schema mismatch: {artifact_path}')
    fitted = load_model_artifact(artifact_path, expected={'feature_columns': list(FEATURE_SET.source_columns)}, trust_artifact=True)
    frame = window_frames[window]
    columns = ['split', 'text_id', 'label', *FEATURE_SET.source_columns]
    scored = score_feature_vector_model(fitted, frame[columns])
    aggregate = scored.groupby('text_id', as_index=False).agg(mg_score=('mg_score', 'mean'), segment_count=('mg_score', 'size'))
    aggregate['prediction'] = (aggregate['mg_score'] >= 0.5).astype('int64')
    details = frame[['text_id', 'id', 'source_id', 'prompt_id', 'source_llm', 'origin', 'source_dataset']].drop_duplicates('text_id')
    return details.merge(aggregate, on='text_id', validate='one_to_one').assign(model_family=model_family, window_configuration=window)

predictions_long = pd.concat(
    [score_configuration(window, model_family) for window, model_family in tqdm(
        itertools.product(WINDOWS, MODEL_FAMILIES), desc='Scoring classifiers', unit='configuration',
    )],
    ignore_index=True,
)
if len(predictions_long) != LIMIT * len(WINDOWS) * len(MODEL_FAMILIES):
    raise ValueError('unexpected prediction row count')
predictions_long.sort_values(['id', 'model_family', 'window_configuration'])

In [ ]:
comparison_summary = (predictions_long.groupby(['model_family', 'window_configuration'], as_index=False).agg(
    text_count=('id', 'nunique'), predicted_mg_count=('prediction', 'sum'),
    mean_mg_score=('mg_score', 'mean'), median_mg_score=('mg_score', 'median'), std_mg_score=('mg_score', 'std'),
)).assign(predicted_mg_fraction=lambda frame: frame['predicted_mg_count'] / frame['text_count'])

disagreement_frames = []
for window in WINDOWS:
    pair = predictions_long.loc[predictions_long['window_configuration'].eq(window)].pivot(index='id', columns='model_family', values='prediction')
    pair['window_configuration'] = window
    pair['disagree'] = pair['logistic_regression'] != pair['random_forest']
    disagreement_frames.append(pair.reset_index())
disagreements = pd.concat(disagreement_frames, ignore_index=True)
window_predictions = predictions_long.pivot(index=['id', 'model_family'], columns='window_configuration', values='prediction')
window_changes = window_predictions.assign(changed_window_count=window_predictions.nunique(axis=1) - 1).reset_index()

mg_results = predictions_long.assign(is_correct_mg=predictions_long['prediction'].eq(1))
aggregate_results = (mg_results.groupby(['model_family', 'window_configuration'], as_index=False).agg(
    text_count=('id', 'nunique'), prediction_count=('id', 'size'), detected_mg_count=('is_correct_mg', 'sum'),
    mg_detection_rate=('is_correct_mg', 'mean'), mean_mg_score=('mg_score', 'mean'),
)).assign(aggregation_scope='window')

if predictions_long["source_llm"].isna().any():
    raise ValueError('source_llm metadata is missing for one or more predictions')

source_llm_accuracy = (
    predictions_long
    .groupby(["source_llm", "model_family", "window_configuration"], as_index=False)
    .agg(
        text_count=("text_id", 'nunique'),
        correct_mg_count=("prediction", lambda values: values.eq(1).sum()),
    )
    .assign(mg_detection_accuracy=lambda frame: frame["correct_mg_count"] / frame["text_count"])
    .sort_values(["source_llm", "model_family", "window_configuration"])
    .reset_index(drop=True)
)
if source_llm_accuracy.empty:
    raise ValueError('source LLM accuracy report is empty')
if not source_llm_accuracy["mg_detection_accuracy"].between(0, 1).all():
    raise ValueError('MG detection accuracy must be between 0 and 1')
print('MG-only detection rate; this is not balanced classifier accuracy.')
display(predictions_long.sort_values(['id', 'model_family', 'window_configuration']))
display(comparison_summary)
display(disagreements)
display(window_changes)
display(aggregate_results)
display(source_llm_accuracy)

In [ ]:
source_llm_accuracy_pivot = source_llm_accuracy.pivot(
    index='source_llm',
    columns=['model_family', 'window_configuration'],
    values='mg_detection_accuracy',
).sort_index(axis=1)
source_llm_processed_texts = predictions_long.groupby('source_llm')['text_id'].nunique()
source_llm_accuracy_pivot.insert(0, ('', 'processed_texts'), source_llm_processed_texts)
display(source_llm_accuracy_pivot)

In [ ]:
prediction_token_counts = processed_texts[['id', 'token_count']].copy()
prediction_token_counts['id'] = prediction_token_counts['id'].astype(str)
predictions_with_token_count = predictions_long.copy()
predictions_with_token_count['id'] = predictions_with_token_count['id'].astype(str)
predictions_with_token_count = predictions_with_token_count.merge(
    prediction_token_counts, on='id', how='left', validate='many_to_one',
)
if predictions_with_token_count['token_count'].isna().any():
    raise ValueError('token counts are missing for one or more predictions')
long_text_predictions = predictions_with_token_count.loc[
    predictions_with_token_count['token_count'].ge(MIN_TOKEN_COUNT)
].copy()
if long_text_predictions.empty:
    raise ValueError(f'no texts meet token-count lower bound {MIN_TOKEN_COUNT}')

long_text_source_accuracy = (
    long_text_predictions
    .groupby(['source_llm', 'model_family', 'window_configuration'], as_index=False)
    .agg(
        text_count=('text_id', 'nunique'),
        correct_mg_count=('prediction', lambda values: values.eq(1).sum()),
    )
    .assign(mg_detection_accuracy=lambda frame: frame['correct_mg_count'] / frame['text_count'])
)
long_text_aggregate_accuracy = (
    long_text_predictions
    .groupby(['model_family', 'window_configuration'], as_index=False)
    .agg(
        text_count=('text_id', 'nunique'),
        correct_mg_count=('prediction', lambda values: values.eq(1).sum()),
    )
    .assign(
        source_llm='ALL',
        mg_detection_accuracy=lambda frame: frame['correct_mg_count'] / frame['text_count'],
    )
)
long_text_accuracy = pd.concat([long_text_source_accuracy, long_text_aggregate_accuracy], ignore_index=True)
long_text_accuracy_pivot = long_text_accuracy.pivot(
    index='source_llm',
    columns=['model_family', 'window_configuration'],
    values='mg_detection_accuracy',
).sort_index(axis=1)
long_text_processed_texts = long_text_predictions.groupby('source_llm')['text_id'].nunique()
long_text_processed_texts.loc['ALL'] = long_text_predictions['text_id'].nunique()
long_text_accuracy_pivot.insert(0, ('', 'processed_texts'), long_text_processed_texts)
display(long_text_accuracy_pivot)

## Logistic-regression accuracy by minimum text length

Show MG detection accuracy for each source LLM after progressively excluding texts shorter than the selected token threshold.

In [ ]:
MIN_TOKEN_THRESHOLD = 50
TOKEN_THRESHOLD_STEP = 10
LENGTH_WINDOWS = (32, 64)

length_predictions = predictions_with_token_count.loc[
    predictions_with_token_count['model_family'].eq('logistic_regression')
    & predictions_with_token_count['window_configuration'].isin(LENGTH_WINDOWS)
].copy()

max_token_count = int(length_predictions['token_count'].max())
token_thresholds = range(
    MIN_TOKEN_THRESHOLD,
    max_token_count + 1,
    TOKEN_THRESHOLD_STEP,
)
source_llms = sorted(length_predictions['source_llm'].unique())
source_llm_colors = dict(zip(source_llms, sns.color_palette('husl', n_colors=len(source_llms))))

length_accuracy_frames = []
for min_token_count in token_thresholds:
    eligible = length_predictions.loc[
        length_predictions['token_count'].ge(min_token_count)
    ]
    length_accuracy_frames.append(
        eligible.assign(min_token_count=min_token_count).groupby(
            ['window_configuration', 'source_llm', 'min_token_count'],
            as_index=False,
        ).agg(
            text_count=('prediction', 'size'),
            mg_detection_accuracy=('prediction', lambda values: values.eq(1).mean()),
        )
    )

source_llm_length_accuracy = pd.concat(length_accuracy_frames, ignore_index=True)

fig, axes = plt.subplots(
    len(LENGTH_WINDOWS), 1,
    figsize=(16, 5 * len(LENGTH_WINDOWS)),
    sharex=True,
    sharey=True,
)
for axis, window in zip(axes, LENGTH_WINDOWS):
    sns.lineplot(
        data=source_llm_length_accuracy.loc[
            source_llm_length_accuracy['window_configuration'].eq(window)
        ],
        x='min_token_count',
        y='mg_detection_accuracy',
        hue='source_llm',
        palette=source_llm_colors,
        marker='o',
        ax=axis,
    )
    axis.set_title(f'Logistic regression / {window}-token window')
    axis.set_xlabel('Minimum text length (tokens)')
    axis.set_ylabel('MG detection accuracy')
    axis.set_ylim(0, 1)
    axis.grid(axis='both', alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
for axis in axes:
    axis.get_legend().remove()
fig.legend(handles, labels, title='Source LLM', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout(rect=(0, 0, 0.88, 1))
save_png_figure(fig, PLOT_OUTPUT_ROOT / 'logistic_accuracy_by_minimum_text_length.png')
plt.show()

In [ ]:
source_llms = sorted(long_text_predictions['source_llm'].dropna().unique())
configurations = [
    ('logistic_regression', 32),
    ('logistic_regression', 64),
    ('random_forest', 32),
    ('random_forest', 64),
]

source_llm_counts = (
    long_text_predictions
    .groupby(['source_llm', 'model_family', 'window_configuration'], as_index=False)
    .agg(
        processed_text_count=('text_id', 'nunique'),
        correct_text_count=('prediction', lambda values: values.eq(1).sum()),
    )
    .assign(
        incorrect_text_count=lambda frame: (
            frame['processed_text_count'] - frame['correct_text_count']
        )
    )
)
configuration_index = pd.MultiIndex.from_product(
    [source_llms, ['logistic_regression', 'random_forest'], [32, 64]],
    names=['source_llm', 'model_family', 'window_configuration'],
)
source_llm_counts = (
    source_llm_counts
    .set_index(['source_llm', 'model_family', 'window_configuration'])
    .reindex(configuration_index)
    .fillna(0)
    .reset_index()
)

colors = {
    'correct': 'seagreen',
    'incorrect': 'lightgray',
}
fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharex=True, sharey=True)
for axis, (model_family, window) in zip(axes.flat, configurations):
    panel = source_llm_counts.loc[
        source_llm_counts['model_family'].eq(model_family)
        & source_llm_counts['window_configuration'].eq(window)
    ].set_index('source_llm').reindex(source_llms).reset_index()
    positions = range(len(source_llms))
    axis.bar(
        positions,
        panel['correct_text_count'],
        color=colors['correct'],
        label='Correctly classified MG texts',
    )
    axis.bar(
        positions,
        panel['incorrect_text_count'],
        bottom=panel['correct_text_count'],
        color=colors['incorrect'],
        label='Incorrectly classified MG texts',
    )
    axis.set_title(f'{model_family} / {window}-token window')
    axis.set_xlabel('Source LLM')
    axis.set_ylabel('Processed texts')
    axis.set_xticks(list(positions), source_llms, rotation=90)
    axis.grid(axis='y', alpha=0.25)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2)
fig.suptitle(f'Classification counts per source LLM (token_count >= {MIN_TOKEN_COUNT})')
plt.tight_layout(rect=(0, 0, 1, 0.94))
save_png_figure(fig, PLOT_OUTPUT_ROOT / 'classification_counts_by_source_llm.png')
plt.show()

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
sns.lineplot(data=comparison_summary, x='window_configuration', y='mean_mg_score', hue='model_family', marker='o', ax=axes[0])
axes[0].set(title='Mean MG score', xlabel='Feature window', ylabel='Mean MG probability')
sns.lineplot(data=comparison_summary, x='window_configuration', y='predicted_mg_fraction', hue='model_family', marker='o', ax=axes[1])
axes[1].set(title='Predicted MG fraction', xlabel='Feature window', ylabel='Fraction predicted MG')
plt.tight_layout()
save_png_figure(fig, PLOT_OUTPUT_ROOT / 'classifier_summary.png')
plt.show()

heatmap_frame = predictions_long.assign(configuration=predictions_long['model_family'] + ' / ' + predictions_long['window_configuration'].astype(str)).pivot(index='id', columns='configuration', values='mg_score')
plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_frame, cmap='viridis', vmin=0, vmax=1)
plt.title('Name That LLM MG scores by text, classifier, and window')
plt.tight_layout()
save_png_figure(plt.gcf(), PLOT_OUTPUT_ROOT / 'classifier_score_heatmap.png')
plt.show()

export_root = DATA_ROOT / 'experiment-results' / 'classification' / 'name-that-llm'
export_root.mkdir(parents=True, exist_ok=True)
predictions_long.to_csv(export_root / 'name_that_llm_classifier_predictions_long.csv', index=False)
predictions_long.pivot(index=['id', 'source_id', 'prompt_id', 'source_llm'], columns=['model_family', 'window_configuration'], values=['mg_score', 'prediction', 'segment_count']).to_csv(export_root / 'name_that_llm_classifier_predictions_wide.csv')
print(f'Wrote external results to {export_root}')